# Stellar Classification Pipeline
This notebook will guide us through:
1. Data Acquisition
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Training and Evaluation

In [ ]:
import os
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Display plots inline
%matplotlib inline

## 1. Data Acquisition
First, we will ensure that our dataset is downloaded.

In [ ]:
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)
csv_file = os.path.join(data_dir, 'Stars.csv')

if not os.path.exists(csv_file):
    print("Downloading Stars.csv...")
    url = "https://github.com/YBIFoundation/Dataset/raw/main/Stars.csv"
    urllib.request.urlretrieve(url, csv_file)
    print("Download complete.")
else:
    print("Stars.csv already exists.")

## 2. Exploratory Data Analysis (EDA)
Let's load the data using pandas and inspect its structure.

In [ ]:
df = pd.read_csv(csv_file)
print(f"Dataset Shape: {df.shape}")
df.head()

We will now plot the correlation matrix for numerical features and the distribution of our target variable (`Star type`).

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
numerical_df = df.select_dtypes(include=['number'])
sns.heatmap(numerical_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()

# Class Distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Star type', palette='viridis')
plt.title('Distribution of Star Types')
plt.show()

## 3. Data Preprocessing
Machine learning models require features to be encoded and scaled appropriately. We will build a preprocessing pipeline to handle categorical and numerical columns separately.

In [ ]:
X = df.drop(['Star type', 'Star category'], axis=1, errors='ignore')
y = df['Star type']

categorical_cols = ['Star color', 'Spectral Class']
numerical_cols = ['Temperature (K)', 'Luminosity (L/Lo)', 'Radius (R/Ro)', 'Absolute magnitude (Mv)']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## 4. Model Training & Evaluation
We will train two robust models: **Random Forest** and **Support Vector Machine (SVM)**, then evaluate their performance using a classification report and a confusion matrix.

In [ ]:
# Model 1: Random Forest
print("--- Random Forest ---")
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', RandomForestClassifier(random_state=42))])
rf_pipeline.fit(X_train, y_train)
rf_predictions = rf_pipeline.predict(X_test)

print(classification_report(y_test, rf_predictions))

cm_rf = confusion_matrix(y_test, rf_predictions)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
disp_rf.plot(cmap='Blues')
plt.title('Random Forest Confusion Matrix')
plt.show()

In [ ]:
# Model 2: Support Vector Machine (SVM)
print("\n--- Support Vector Machine (SVM) ---")
svm_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', SVC(random_state=42))])
svm_pipeline.fit(X_train, y_train)
svm_predictions = svm_pipeline.predict(X_test)

print(classification_report(y_test, svm_predictions))

cm_svm = confusion_matrix(y_test, svm_predictions)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm)
disp_svm.plot(cmap='Blues')
plt.title('SVM Confusion Matrix')
plt.show()